[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/monacofj/misda/blob/main/examples/diagnostic_noisy.ipynb)

# MISDA — noisy controlled diagnostics

This notebook evaluates the same unified controlled diagnostic suite as `diagnostic_clean.ipynb`, but after a reproducible observation process adds scale-relative noise. Ground truth remains defined by the theoretical problem and the sampled clean objectives `Z`; MISDA receives only the observed matrix `Y`. The fixed `sigma=0.10` condition is a reference observation regime, not a claimed robustness threshold.

In [ ]:
from pathlib import Path
import subprocess
import sys

# In a repository checkout, test the local code. In Colab, install main.
target = ".[benchmarks]" if Path("pyproject.toml").exists() else "git+https://github.com/monacofj/misda.git@main#egg=misda[benchmarks]"
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", target])


In [ ]:
import misda
import misda.benchmarks as bench

N = 300
SEED = 123
OBSERVATION_SEED = 456
SIGMA = 0.10

## Scale-relative observation noise

For each clean objective `Z_j`, observation follows `Y_j = Z_j + sigma * std(Z_j) * epsilon_j`, with `epsilon_j ~ N(0,1)`. Sampling and observation use distinct fixed seeds. Thus `sigma=0.10` means an observation-noise standard deviation equal to 10% of that clean objective's sample standard deviation. This is a reproducible reference condition only; degradation across noise levels is studied separately in `diagnostic_robustness.ipynb`.

In [ ]:
noisy_results = {}
for problem in bench.PROBLEMS:
    dataset = problem.generate(
        N=N,
        seed=SEED,
        sigma=SIGMA,
        observation_seed=OBSERVATION_SEED,
    )
    truth = bench.diagnostic_truth(problem, dataset.Z)
    mis_set = misda.discover(dataset.Y, name=truth["name"], seed=SEED)
    misda.evaluate(mis_set, metrics=("linear", "pareto"))
    benchmark_result = misda.benchmark(mis_set, truth)
    print(benchmark_result.report())
    mis_set.graph_plot()
    noisy_results[problem.id] = {
        "dataset": dataset,
        "result_obj": mis_set,
        "benchmark_obj": benchmark_result,
        "truth": truth,
    }